In [1]:
import csv
from collections import defaultdict
from typing import Dict, List, Tuple

CSV_PATH = "ucl_2425_league_phase_results.csv"

#load match data
def load_matches(csv_path: str) -> List[dict]:
    matches = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            matches.append({
                "matchday": int(row["matchday"]),
                "home": row["home"],
                "away": row["away"],
                "hg": int(row["home_goals"]),
                "ag": int(row["away_goals"]),
            })
    return matches

#points and relations
def outcome_points(goals_for: int, goals_against: int) -> int:
    if goals_for > goals_against:
        return 3
    elif goals_for == goals_against:
        return 1
    else:
        return 0


def build_points_and_relations(
    matches: List[dict],
) -> Tuple[Dict[str, int], Dict[str, list], Dict[str, list]]:
    
    points = defaultdict(int)
    beat   = defaultdict(list)
    drew   = defaultdict(list)

    for m in matches:
        h, a   = m["home"], m["away"]
        hg, ag = m["hg"],   m["ag"]

        points[h] += outcome_points(hg, ag)
        points[a] += outcome_points(ag, hg)

        if hg > ag:
            beat[h].append(a)
        elif ag > hg:
            beat[a].append(h)
        else:
            drew[h].append(a)
            drew[a].append(h)

    return dict(points), dict(beat), dict(drew)



def iterate_ratings(
    points: Dict[str, int],
    beat:   Dict[str, list],
    drew:   Dict[str, list],
    beta:           float = 0.15,
    gamma:          float = 0.5,
    num_iterations: int   = 30,
) -> Dict[str, float]:
    teams  = list(points.keys())
    rating = {t: float(points[t]) for t in teams}

    for _ in range(num_iterations):
        new_rating = {}
        for t in teams:
            beaten_contrib = sum(rating[opp] for opp in beat.get(t, []))
            drawn_contrib  = sum(rating[opp] for opp in drew.get(t, []))
            new_rating[t]  = (
                points[t]
                + beta * (beaten_contrib + gamma * drawn_contrib)
            )
        rating = new_rating

    return rating


#ranking and analysis utilities
def get_rank(scored: Dict[str, float]) -> Dict[str, int]:
    rows = sorted(scored.items(), key=lambda x: (-x[1], x[0]))
    return {team: r + 1 for r, (team, _) in enumerate(rows)}


def kendall_tau(rank_a: Dict[str, int], rank_b: Dict[str, int]) -> float:
    teams = list(rank_a.keys())
    n = len(teams)
    c = d = 0
    for p in range(n):
        for q in range(p + 1, n):
            t1, t2 = teams[p], teams[q]
            prod = (rank_a[t1] - rank_a[t2]) * (rank_b[t1] - rank_b[t2])
            if prod > 0:   c += 1
            elif prod < 0: d += 1
    return (c - d) / (n * (n - 1) // 2)


def spearman_rho(rank_a: Dict[str, int], rank_b: Dict[str, int]) -> float:
    teams = list(rank_a.keys())
    n = len(teams)
    d_sq = sum((rank_a[t] - rank_b[t]) ** 2 for t in teams)
    return 1.0 - (6.0 * d_sq) / (n * (n * n - 1))


def topk_overlap(rank_a: Dict[str, int], rank_b: Dict[str, int], k: int) -> int:
    top_a = {t for t, r in rank_a.items() if r <= k}
    top_b = {t for t, r in rank_b.items() if r <= k}
    return len(top_a & top_b)


def print_ranking_table(
    rat_sorted:  List[Tuple[str, float]],
    points:      Dict[str, int],
    std_rank:    Dict[str, int],
    rat_rank:    Dict[str, int],
    beta:        float,
    gamma:       float,
    num_iters:   int,
) -> None:
    print(f"Parameters: beta={beta}, gamma={gamma}, iterations={num_iters}")
    print(f"{'Pos':>3}  {'Team':25}  {'Pts':>4}  {'Rating':>10}  {'Delta':>6}")
    print("-" * 60)
    for pos, (team, r) in enumerate(rat_sorted, start=1):
        pts   = points[team]
        delta = std_rank[team] - rat_rank[team]
        print(f"{pos:>3}  {team:25}  {pts:>4}  {r:>10.3f}  {delta:>+6}")
    print()


def main():
    matches = load_matches(CSV_PATH)
    points, beat, drew = build_points_and_relations(matches)

    #standard points table (baseline)
    std_rank = get_rank({t: float(v) for t, v in points.items()})

    #parameters
    beta           = 0.15
    gamma          = 0.5
    num_iterations = 30

    #compute ratings
    rating   = iterate_ratings(points, beat, drew, beta, gamma, num_iterations)
    rat_rank = get_rank(rating)
    rat_sorted = sorted(rating.items(), key=lambda x: (-x[1], x[0]))

    #print ranking table
    print_ranking_table(rat_sorted, points, std_rank, rat_rank,
                        beta, gamma, num_iterations)

    #eank correlation vs standard points table
    tau = kendall_tau(std_rank, rat_rank)
    rho = spearman_rho(std_rank, rat_rank)

    print("=" * 50)
    print("Rank correlation vs standard points table")
    print("=" * 50)
    print(f"  Kendall's tau:  {tau:+.4f}")
    print(f"  Spearman's rho: {rho:+.4f}")
    print()

    # Top-k overlap
    print("Top-k overlap with standard points table")
    print(f"  Top  8 (auto qualification): {topk_overlap(std_rank, rat_rank,  8)} / 8")
    print(f"  Top 24 (play-off cut):       {topk_overlap(std_rank, rat_rank, 24)} / 24")
    print()


if __name__ == "__main__":
    main()

Parameters: beta=0.15, gamma=0.5, iterations=30
Pos  Team                        Pts      Rating   Delta
------------------------------------------------------------
  1  Liverpool                    21      42.594      +0
  2  Barcelona                    19      37.869      +1
  3  Arsenal                      19      36.950      -1
  4  Lille                        16      33.681      +4
  5  Inter Milan                  19      33.287      -1
  6  Bayer Leverkusen             16      31.876      +1
  7  Real Madrid                  15      29.604      +6
  8  Atlético Madrid              18      28.895      -3
  9  Aston Villa                  16      27.932      -3
 10  Benfica                      13      27.776      +5
 11  Monaco                       13      27.737      +7
 12  PSV Eindhoven                14      27.148      +2
 13  Bayern Munich                15      26.995      -3
 14  Atalanta                     15      26.970      -5
 15  Borussia Dortmund            15